<a href="https://colab.research.google.com/github/tijanicica/hello-baby-knowledge-graph/blob/ner-preparation/ner_bertic_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Faza 2.2: dotreniravanje BERTić-a za NER (Halo beba)

**Pre pokretanja:** `Runtime → Change runtime type → T4 GPU`.

**Ulaz:** folder `data_ner/` koji pravi `ner_priprema.py` (`train.jsonl`, `dev.jsonl`, `test.jsonl`, `labels.json`, `split.json`).
Lokalno ga spakuj: `zip -r data_ner.zip data_ner` i otpremi u ćeliji ispod.

**Napomena o podacima:** otpremanjem u Colab tekst dijaloga ide na Google servere. Proveri sa mentorom da li je to dozvoljeno za ovu bazu.

**Sadržaj:**
1. Instalacija i učitavanje podataka
2. Bazni model (rečnik iz trening skupa), za poređenje
3. Dotreniravanje BERTić-a na fiksnoj podeli (105 / 15 / 30 dijaloga)
4. Rezultati po tipu entiteta i analiza grešaka
5. Unakrsna validacija u 5 foldova (glavni rezultat za tezu, traje duže)

## 1. Instalacija i podaci

In [5]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NEMA GPU - uključi T4 u Runtime podešavanjima")

GPU: Tesla T4


In [6]:
# Otpremi data_ner.zip (napravljen lokalno sa: zip -r data_ner.zip data_ner).
# Ako je fajl već otpremljen (npr. prevučen u panel sa fajlovima), ovaj korak se preskače.
import os
from google.colab import files
if not os.path.exists("data_ner.zip"):
    files.upload()
!unzip -o -q data_ner.zip
!ls data_ner

# Alternativa: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/data_ner .

dev.jsonl  labels.json	split.json  test.jsonl	train.jsonl


In [7]:
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data_ner")
MODEL_NAME = "classla/bcms-bertic"

# Podešavanja treniranja
LEARNING_RATE = 3e-5
EPOCHS = 10
BATCH_SIZE = 16
SEED = 42
USE_CLASS_WEIGHTS = True   # True: greška na retkom tipu košta više (pomaže tipovima sa F1 = 0)
FIX_BIO = False              # popravka BIO niza posle predviđanja: I-X bez B-X postaje B-X
BEST_METRIC = "macro_f1"    # po čemu se bira najbolja epoha: "macro_f1" (prosek po tipovima) ili "f1" (micro)

records = [json.loads(line) for name in ["train", "dev", "test"]
           for line in open(DATA / f"{name}.jsonl", encoding="utf-8")]
label_list = json.loads((DATA / "labels.json").read_text(encoding="utf-8"))
split = json.loads((DATA / "split.json").read_text(encoding="utf-8"))

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}


def select(dialog_ids):
    ids = set(dialog_ids)
    return [r for r in records if r["dialog_id"] in ids]


train_rows, dev_rows, test_rows = select(split["train"]), select(split["dev"]), select(split["test"])
print(f"Replika: train {len(train_rows)}, dev {len(dev_rows)}, test {len(test_rows)}; oznaka: {len(label_list)}")

Replika: train 2856, dev 419, test 843; oznaka: 61


## 2. Bazni model: rečnik iz trening skupa

Svi entiteti iz trening skupa se zapamte (tekst → najčešći tip), pa se u test tekstu traže najduži pogoci.
Ovo pokazuje koliko se postiže samim pamćenjem; BERTić treba da bude bolji, posebno na oblicima koje nije video.

Evaluacija je stroga, na nivou entiteta (isto pravilo kao `seqeval` u strogom IOB2 režimu): entitet se računa kao tačan samo ako se poklapaju i granice i tip. Funkcije su napisane ovde, bez dodatnih biblioteka.

In [8]:
def spans_of(tokens, tags):
    """Entiteti iz BIO oznaka kao (početak, kraj, tip), po strogom IOB2 pravilu:
    entitet počinje sa B-X, nastavlja se samo sa I-X istog tipa; I- bez B- se ne računa."""
    spans, start, label = [], None, None
    for i, tag in enumerate(tags + ["O"]):
        if start is not None and tag != "I-" + label:
            spans.append((start, i, label))
            start = None
        if tag.startswith("B-"):
            start, label = i, tag[2:]
    return spans


def fix_bio(tags):
    """Popravka BIO niza: I-X koji ne nastavlja entitet istog tipa (dolazi posle O
    ili posle drugog tipa) pretvara se u B-X, tj. započinje novi entitet.
    Bez ovoga stroga evaluacija takav niz potpuno odbacuje."""
    fixed = []
    for tag in tags:
        if tag.startswith("I-"):
            prev = fixed[-1] if fixed else "O"
            if prev == "O" or prev[2:] != tag[2:]:
                tag = "B-" + tag[2:]
        fixed.append(tag)
    return fixed


def entity_counts(gold, pred):
    """Brojanje po tipu entiteta, za strogu i blagu evaluaciju.
    stroga: entitet je tačan ako se poklapaju i granice i tip (tp);
    blaga:  entitet je pogođen ako se preklapa sa entitetom istog tipa
            (gold_hit = pronađeni ručni entiteti, pred_hit = tačne predikcije)."""
    counts = {}
    for g_tags, p_tags in zip(gold, pred):
        g, p = set(spans_of(None, g_tags)), set(spans_of(None, p_tags))
        for s, e, lab in g:
            c = counts.setdefault(lab, Counter())
            c["gold"] += 1
            c["gold_hit"] += any(lab == l2 and s < e2 and e > s2 for s2, e2, l2 in p)
        for s, e, lab in p:
            c = counts.setdefault(lab, Counter())
            c["pred"] += 1
            c["pred_hit"] += any(lab == l2 and s < e2 and e > s2 for s2, e2, l2 in g)
        for _, _, lab in g & p:
            counts[lab]["tp"] += 1
    return counts


def prf(hit_pred, n_pred, hit_gold, n_gold):
    precision = hit_pred / n_pred if n_pred else 0.0
    recall = hit_gold / n_gold if n_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def scores(gold, pred):
    """micro (svi entiteti zajedno), macro (prosek F1 po tipovima) i blagi F1."""
    counts = entity_counts(gold, pred)
    c = sum(counts.values(), Counter())
    p, r, f = prf(c["tp"], c["pred"], c["tp"], c["gold"])
    per_type_f1 = [prf(x["tp"], x["pred"], x["tp"], x["gold"])[2] for x in counts.values() if x["gold"]]
    return {"precision": p, "recall": r, "f1": f,
            "macro_f1": sum(per_type_f1) / len(per_type_f1) if per_type_f1 else 0.0,
            "f1_blago": prf(c["pred_hit"], c["pred"], c["gold_hit"], c["gold"])[2]}


def entity_report(gold, pred):
    """Rezultati po tipu entiteta: {tip: {precision, recall, f1-score, f1-blago, support, predvidjeno}}."""
    report = {}
    for lab, c in sorted(entity_counts(gold, pred).items()):
        if c["gold"] == 0:
            continue  # tip koji se javlja samo u predikcijama, a ne u ground truth
        p, r, f = prf(c["tp"], c["pred"], c["tp"], c["gold"])
        report[lab] = {"precision": p, "recall": r, "f1-score": f,
                       "f1-blago": prf(c["pred_hit"], c["pred"], c["gold_hit"], c["gold"])[2],
                       "support": c["gold"], "predvidjeno": c["pred"]}
    return report


def build_dictionary(rows):
    votes = {}
    for r in rows:
        words = [t.lower() for t in r["tokens"]]
        for s, e, label in spans_of(r["tokens"], r["ner_tags"]):
            votes.setdefault(tuple(words[s:e]), Counter())[label] += 1
    return {k: v.most_common(1)[0][0] for k, v in votes.items()}


def dictionary_predict(dictionary, tokens, max_len=8):
    words = [t.lower() for t in tokens]
    tags, i = ["O"] * len(tokens), 0
    while i < len(tokens):
        for n in range(min(max_len, len(tokens) - i), 0, -1):  # najduži pogodak prvo
            label = dictionary.get(tuple(words[i:i + n]))
            if label:
                tags[i:i + n] = ["B-" + label] + ["I-" + label] * (n - 1)
                i += n
                break
        else:
            i += 1
    return tags


dictionary = build_dictionary(train_rows)
gold_test = [r["ner_tags"] for r in test_rows]
baseline_pred = [dictionary_predict(dictionary, r["tokens"]) for r in test_rows]
baseline = scores(gold_test, baseline_pred)
print(f"Rečnik: {len(dictionary)} izraza")
print("Bazni model (test):", {k: round(v, 3) for k, v in baseline.items()})

Rečnik: 3895 izraza
Bazni model (test): {'precision': 0.165, 'recall': 0.273, 'f1': 0.206, 'macro_f1': 0.225, 'f1_blago': 0.357}


## 3. Dotreniravanje BERTić-a

Dva podešavanja iz ćelije sa parametrima menjaju treniranje:
- `USE_CLASS_WEIGHTS = True`: greška na retkoj oznaci košta više (težina ~ 1/√učestalost). Pokušaj kada retki tipovi imaju F1 = 0.
- `BEST_METRIC`: po kojoj meri se bira najbolja epoha na dev skupu; `"macro_f1"` daje prednost modelu koji pokriva i retke tipove.
- `FIX_BIO = True`: posle predviđanja, oznaka I-X koja ne nastavlja entitet istog tipa postaje B-X. Model ponekad započne entitet sa I- umesto sa B-, a stroga evaluacija takav niz potpuno odbacuje.

In [9]:
import torch
from datasets import Dataset
from transformers import (AutoModelForTokenClassification, AutoTokenizer,
                          DataCollatorForTokenClassification, Trainer, TrainingArguments)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorForTokenClassification(tokenizer)


def encode(rows):
    """Deli reči na delove; oznaku dobija samo prvi deo svake reči, ostali -100 (ignorišu se)."""
    enc = tokenizer([r["tokens"] for r in rows], is_split_into_words=True,
                    truncation=True, max_length=512)
    all_labels = []
    for i, r in enumerate(rows):
        labels, previous = [], None
        for word_id in enc.word_ids(i):
            if word_id is None or word_id == previous:
                labels.append(-100)
            else:
                labels.append(label2id[r["ner_tags"][word_id]])
            previous = word_id
        all_labels.append(labels)
    enc["labels"] = all_labels
    return Dataset.from_dict(dict(enc))


def to_tags(logits, label_ids):
    """Iz izlaza modela vraća BIO oznake po reči (samo pozicije prvih delova reči)."""
    pred_ids = np.argmax(logits, axis=-1)
    gold, pred = [], []
    for p_row, l_row in zip(pred_ids, label_ids):
        keep = l_row != -100
        gold.append([id2label[i] for i in l_row[keep]])
        pred.append([id2label[i] for i in p_row[keep]])
    return gold, pred


def compute_metrics(eval_pred):
    gold, pred = to_tags(eval_pred.predictions, eval_pred.label_ids)
    if FIX_BIO:
        pred = [fix_bio(p) for p in pred]
    return scores(gold, pred)


def class_weights(rows):
    """Težina oznake ~ 1/sqrt(učestalost): retke oznake dobijaju veću težinu, ograničeno na 10."""
    counts = Counter(t for r in rows for t in r["ner_tags"])
    total = sum(counts.values())
    w = [min(10.0, (total / (len(label_list) * counts[l])) ** 0.5) if counts[l] else 10.0
         for l in label_list]
    return torch.tensor(w, dtype=torch.float)


class WeightedTrainer(Trainer):
    """Trainer kod koga se greška na svakoj oznaci množi njenom težinom."""
    def __init__(self, *args, weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.weights = weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)), labels.view(-1),
            weight=self.weights.to(logits.device), ignore_index=-100)
        return (loss, outputs) if return_outputs else loss


def train_and_test(train_rows, dev_rows, test_rows, out_dir, seed=SEED):
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=out_dir,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32,
        weight_decay=0.01,
        # zagrevanje: prvih 10% koraka learning rate raste od 0 (ceo broj radi u svim verzijama)
        warmup_steps=int(0.1 * EPOCHS * -(-len(train_rows) // BATCH_SIZE)),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_only_model=True,            # ne čuva stanje optimizatora: brže čuvanje posle epohe
        save_total_limit=1,
        load_best_model_at_end=True,     # zadržava najbolju epohu na dev skupu
        metric_for_best_model=BEST_METRIC,
        seed=seed,
        report_to="none",
        logging_steps=50,
    )
    common = dict(model=model, args=args, train_dataset=encode(train_rows),
                  eval_dataset=encode(dev_rows), data_collator=collator,
                  processing_class=tokenizer, compute_metrics=compute_metrics)
    if USE_CLASS_WEIGHTS:
        trainer = WeightedTrainer(**common, weights=class_weights(train_rows))
    else:
        trainer = Trainer(**common)
    trainer.train()
    output = trainer.predict(encode(test_rows))
    gold, pred = to_tags(output.predictions, output.label_ids)
    return trainer, gold, pred

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

In [10]:
trainer, gold, pred_raw = train_and_test(train_rows, dev_rows, test_rows, out_dir="bertic_ner")
pred_fixed = [fix_bio(p) for p in pred_raw]
pred = pred_fixed if FIX_BIO else pred_raw   # ovo koriste sve sledeće ćelije

summary = pd.DataFrame({
    "Bazni model (rečnik)": baseline,
    "BERTić (sirov izlaz)": scores(gold, pred_raw),
    "BERTić (popravljen BIO)": scores(gold, pred_fixed),
}).T.round(3)
bertic = scores(gold, pred)
print(f"Težine klasa: {'DA' if USE_CLASS_WEIGHTS else 'NE'}, izbor epohe po: {BEST_METRIC}, "
      f"popravka BIO: {'DA' if FIX_BIO else 'NE'}")
summary

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForTokenClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.bias                                   | MISSING    | 
classifier.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Težine klasa: DA, izbor epohe po: macro_f1, popravka BIO: NE


,precision,recall,f1,macro_f1,f1_blago
Bazni model (rečnik),0.165,0.273,0.206,0.225,0.357
BERTić (sirov izlaz),0.357,0.468,0.405,0.353,0.646
BERTić (popravljen BIO),0.323,0.478,0.386,0.338,0.643


## 4. Rezultati po tipu i analiza grešaka

- **F1_bertic**: stroga mera, entitet je tačan samo ako se poklapaju granice i tip.
- **F1_blago**: entitet je pogođen ako se preklapa sa ručnim entitetom istog tipa ("crvenilo po vratu" i "neko crvenilo po vratu"). Za graf je ovo važnije, jer se tekst ionako normalizuje.
- **macro F1**: prosek F1 po tipovima; svaki tip ima istu težinu, pa retki tipovi sa F1 = 0 obaraju prosek.
- **predvidjeno**: koliko entiteta tog tipa je model predvideo; 0 znači da tip nikad ne koristi.
- **F1_sirovo**: stroga F1 pre popravke BIO niza; razlika prema **F1_bertic** pokazuje koliko je popravka pomogla.

In [11]:
report = entity_report(gold, pred)
raw_report = entity_report(gold, pred_raw)
base_report = entity_report(gold_test, baseline_pred)
per_type = pd.DataFrame({
    "broj_u_testu": {t: report[t]["support"] for t in report},
    "predvidjeno": {t: report[t]["predvidjeno"] for t in report},
    "F1_recnik": {t: base_report.get(t, {}).get("f1-score", 0.0) for t in report},
    "precision": {t: report[t]["precision"] for t in report},
    "recall": {t: report[t]["recall"] for t in report},
    "F1_sirovo": {t: raw_report[t]["f1-score"] for t in report},
    "F1_bertic": {t: report[t]["f1-score"] for t in report},
    "F1_blago": {t: report[t]["f1-blago"] for t in report},
}).sort_values("broj_u_testu", ascending=False).round(3)
per_type

,broj_u_testu,predvidjeno,F1_recnik,precision,recall,F1_sirovo,F1_bertic,F1_blago
SYMPTOM,201,222,0.317,0.586,0.647,0.615,0.615,0.785
BEHAVIOR,189,275,0.158,0.316,0.460,0.375,0.375,0.711
INTERVENTION,149,183,0.078,0.311,0.383,0.343,0.343,0.712
TIME,145,197,0.228,0.350,0.476,0.404,0.404,0.571
UNCERTAINTY,99,170,0.116,0.359,0.616,0.454,0.454,0.656
FEEDING,86,122,0.314,0.361,0.512,0.423,0.423,0.730
ENVIRONMENT,81,104,0.131,0.250,0.321,0.281,0.281,0.596
EVENT,81,81,0.118,0.358,0.358,0.358,0.358,0.531
DURATION,76,99,0.244,0.323,0.421,0.366,0.366,0.613
BODY_PART,75,77,0.327,0.532,0.547,0.539,0.539,0.651


In [12]:
# Dijagnostika: koje tipove model nikad ne predviđa i koliko grešaka su samo granice
train_counts = Counter(t[2:] for r in train_rows for t in r["ner_tags"] if t.startswith("B-"))
pred_tokens = Counter(t[2:] for p in pred_raw for t in p if t != "O")
diag = pd.DataFrame({
    "u_treningu": {t: train_counts[t] for t in report},
    "u_testu": {t: report[t]["support"] for t in report},
    "entiteta_sirovo": {t: raw_report[t]["predvidjeno"] for t in report},
    "entiteta_popravljeno": {t: report[t]["predvidjeno"] for t in report},
    "predvidjeno_tokena": {t: pred_tokens[t] for t in report},
}).sort_values("u_treningu")
never = diag.index[diag["predvidjeno_tokena"] == 0].tolist()
print(f"Tipovi koje model NIJEDNOM nije predvideo u testu ({len(never)}): {', '.join(never) or '-'}")
print(f"\nStroga F1 = {bertic['f1']:.3f}, blaga F1 = {bertic['f1_blago']:.3f}  "
      f"-> razlika su entiteti nađeni sa pogrešnim granicama")
print(f"Micro F1 = {bertic['f1']:.3f}, macro F1 = {bertic['macro_f1']:.3f}  "
      f"-> macro je niži jer svaki tip, i retki, ulazi u prosek sa istom težinom")
diag

Tipovi koje model NIJEDNOM nije predvideo u testu (1): MOTHER_NUTRITION

Stroga F1 = 0.405, blaga F1 = 0.646  -> razlika su entiteti nađeni sa pogrešnim granicama
Micro F1 = 0.405, macro F1 = 0.353  -> macro je niži jer svaki tip, i retki, ulazi u prosek sa istom težinom


,u_treningu,u_testu,entiteta_sirovo,entiteta_popravljeno,predvidjeno_tokena
VACCINATION_STATUS,18,2,2,2,2
KINDERGARTEN,28,23,20,20,42
CLOTHES,35,9,11,11,17
MOTHER_NUTRITION,38,4,0,0,0
HYDRATION,43,18,13,13,47
CAREGIVER_SUPPORT,51,13,21,21,35
URINATION,55,16,10,10,18
CHILD_TRAIT,70,28,34,34,71
STOOL,74,16,21,21,21
MEDICAL_DEVICE,78,24,17,17,25


In [13]:
def error_types(rows, gold, pred):
    """Svrstava svaku grešku: pogrešan tip, pogrešne granice, propušten ili lažan entitet."""
    counts, examples = Counter(), []
    for r, g, p in zip(rows, gold, pred):
        gs, ps = set(spans_of(r["tokens"], g)), set(spans_of(r["tokens"], p))
        for s, e, lab in gs - ps:
            same_span = [x for x in ps if x[:2] == (s, e)]
            overlap = [x for x in ps if x[0] < e and x[1] > s]
            if same_span:
                kind = f"pogrešan tip ({lab} → {same_span[0][2]})"
            elif overlap:
                kind = f"pogrešne granice ({lab})"
            else:
                kind = f"propušten ({lab})"
            counts[kind] += 1
            examples.append((kind, " ".join(r["tokens"][s:e]), " ".join(r["tokens"])))
        for s, e, lab in ps - gs:
            if not any(x[0] < e and x[1] > s for x in gs):
                counts[f"lažan ({lab})"] += 1
    return counts, examples


counts, examples = error_types(test_rows, gold, pred)
print("Najčešće greške:")
for kind, n in counts.most_common(20):
    print(f"  {n:>4}  {kind}")

Najčešće greške:
    83  pogrešne granice (BEHAVIOR)
    74  lažan (UNCERTAINTY)
    72  pogrešne granice (INTERVENTION)
    59  lažan (TIME)
    48  pogrešne granice (TIME)
    48  lažan (BEHAVIOR)
    44  pogrešne granice (SYMPTOM)
    36  pogrešne granice (FEEDING)
    35  pogrešne granice (ENVIRONMENT)
    33  lažan (INTERVENTION)
    32  lažan (SYMPTOM)
    31  pogrešne granice (DURATION)
    30  lažan (ENVIRONMENT)
    29  lažan (DURATION)
    28  pogrešne granice (UNCERTAINTY)
    28  lažan (OUTCOME)
    28  pogrešne granice (EVENT)
    26  pogrešne granice (BODY_PART)
    25  lažan (ABNORMALITY)
    25  lažan (FREQUENCY)


In [14]:
# Primeri grešaka (promeni filter da vidiš određeni tip greške)
for kind, span, sentence in [x for x in examples if "pogrešan tip" in x[0]][:10]:
    print(f"{kind}: »{span}«\n    {sentence}\n")

pogrešan tip (BEHAVIOR → FREQUENCY): »noćna buđenja«
    Jesmo , što se tiče jela , on relativno dobro jede i sada , ali smo primetili mnogo češća noćna buđenja .

pogrešan tip (DEGREE → FREQUENCY): »mnogo češća«
    Jesmo , što se tiče jela , on relativno dobro jede i sada , ali smo primetili mnogo češća noćna buđenja .

pogrešan tip (MEDICAL_DEVICE → TREATMENT): »pomoćnih aparata«
    Da li koristite nešto od pomoćnih aparata ?

pogrešan tip (MEDICAL_DEVICE → TREATMENT): »uređajem«
    Izvlačili smo onim uređajem koji se koristi . . . usisivačem , ali nismo koristili taj usisivač baš , već smo više onako ručno usisavali .

pogrešan tip (HYDRATION → CHARACTERISTIC): »tečnost«
    A što se tiče jela , dete odbija hranu , ali ne tečnost ?

pogrešan tip (MEDICAL_DEVICE → INTERVENTION): »aplikatorom«
    Dobro , trebalo bi svakako pre odlaska na spavanje ili u toku dana , pogotovo pre jela , da sipate po par kapi fiziološkog rastvora ili fiziomera ili šta već imate , i onda tim aplikatoro

In [15]:
# Čuvanje predikcija za kasniju analizu i gradnju grafa iz NER-a
with open("predikcije_test.jsonl", "w", encoding="utf-8") as f:
    for r, g, p in zip(test_rows, gold, pred):
        f.write(json.dumps({**r, "pred_tags": p}, ensure_ascii=False) + "\n")
per_type.to_csv("rezultati_po_tipu.csv")
print("Sačuvano: predikcije_test.jsonl, rezultati_po_tipu.csv")

Sačuvano: predikcije_test.jsonl, rezultati_po_tipu.csv


## 5. Unakrsna validacija (5 foldova)

Svaki fold od 30 dijaloga je jednom test skup; od ostalih 120, 15 ide u dev, a 105 u trening.
Rezultat je prosek ± standardna devijacija, pouzdaniji od jedne podele na malom skupu.
Na T4 GPU traje otprilike 5 puta duže od sekcije 3.

In [16]:
cv_rows = []
for k, test_ids in enumerate(split["folds"]):
    rest = [d for d in sorted({r["dialog_id"] for r in records}) if d not in set(test_ids)]
    random.Random(SEED + k).shuffle(rest)
    dev_ids, train_ids = rest[:15], rest[15:]
    tr, dv, te = select(train_ids), select(dev_ids), select(test_ids)

    _, g, p = train_and_test(tr, dv, te, out_dir=f"cv_fold{k}")
    if FIX_BIO:
        p = [fix_bio(x) for x in p]
    base_p = [dictionary_predict(build_dictionary(tr), r["tokens"]) for r in te]
    rep, sc = entity_report(g, p), scores(g, p)
    row = {"fold": k, "F1_bertic": sc["f1"], "macroF1_bertic": sc["macro_f1"],
           "F1blago_bertic": sc["f1_blago"], "F1_recnik": scores(g, base_p)["f1"]}
    row.update({f"tip_{t}": rep[t]["f1-score"] for t in rep})
    cv_rows.append(row)
    print(f"Fold {k}: BERTić F1 = {row['F1_bertic']:.3f}, rečnik F1 = {row['F1_recnik']:.3f}")

cv = pd.DataFrame(cv_rows)
for col in ["F1_bertic", "macroF1_bertic", "F1blago_bertic", "F1_recnik"]:
    print(f"{col:<16} {cv[col].mean():.3f} ± {cv[col].std():.3f}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForTokenClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.bias                                   | MISSING    | 
classifier.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Macro F1,F1 Blago
1,3.549806,3.220204,0.208955,0.052434,0.083832,0.014945,0.136889
2,2.630146,2.422592,0.236381,0.303371,0.265719,0.141533,0.481732
3,2.133888,2.031674,0.273271,0.399501,0.324544,0.211007,0.575568
4,1.673160,1.832450,0.283594,0.429463,0.341609,0.265508,0.592261
5,1.415239,1.703799,0.286432,0.426966,0.342857,0.269091,0.596031
6,1.125334,1.673485,0.322934,0.478152,0.385506,0.307637,0.627458
7,1.010132,1.634841,0.312448,0.466916,0.374374,0.313414,0.612202
8,0.902235,1.629773,0.318777,0.455680,0.375128,0.310039,0.623040
9,0.803127,1.643106,0.328346,0.493134,0.394212,0.328968,0.629229
10,0.792577,1.653507,0.329082,0.483146,0.391502,0.327733,0.630788


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 4: BERTić F1 = 0.441, rečnik F1 = 0.202
F1_bertic        0.415 ± 0.021
macroF1_bertic   0.373 ± 0.012
F1blago_bertic   0.636 ± 0.018
F1_recnik        0.208 ± 0.013


In [17]:
type_cols = [c for c in cv.columns if c.startswith("tip_")]
cv_per_type = pd.DataFrame({"F1_prosek": cv[type_cols].mean(), "F1_std": cv[type_cols].std()})
cv_per_type.index = [c[4:] for c in cv_per_type.index]
cv_per_type = cv_per_type.sort_values("F1_prosek", ascending=False).round(3)
cv.to_csv("cv_po_foldu.csv", index=False)
cv_per_type.to_csv("cv_po_tipu.csv")
cv_per_type

,F1_prosek,F1_std
SYMPTOM,0.585,0.052
STOOL,0.570,0.131
AGE,0.534,0.045
BODY_PART,0.526,0.028
URINATION,0.484,0.279
TREATMENT,0.465,0.098
FOOD,0.446,0.069
FREQUENCY,0.437,0.061
BEHAVIOR,0.433,0.048
TIME,0.432,0.036


## 6. Preuzimanje rezultata i modela

In [19]:
# Rezultati (mali fajlovi)
files.download("rezultati_po_tipu.csv")
files.download("predikcije_test.jsonl")
files.download("cv_po_tipu.csv")
files.download("cv_po_foldu.csv")

# Model (~450 MB) - najbolje sačuvati na Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# trainer.save_model("/content/drive/MyDrive/bertic_ner_halo_beba")
# tokenizer.save_pretrained("/content/drive/MyDrive/bertic_ner_halo_beba")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>